# 01 - Data Collection
**Project:** Ứng dụng Khoa học dữ liệu đa yếu tố trong phân tích, dự đoán và cảnh báo sớm nguy cơ học tập kém của sinh viên
**Bước:** 2 - Data Collection
**Vai trò:** Data Collector / Data Analyst

**Mục tiêu:** Thu thập dữ liệu từ Kaggle, khảo sát cấu trúc dataset, và thiết lập lưu trữ dữ liệu bằng MySQL để chuẩn bị cho các bước làm sạch và phân tích tiếp theo.

**Nguồn dữ liệu:** Student Performance Factors Dataset (Kaggle)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import pandas as pd

## 1. Đọc dữ liệu

Đọc file CSV đã tải từ Kaggle, lưu trong Google Drive.

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Data science/01-Datasheet/StudentPerformanceFactors.csv')
print('Đã đọc dữ liệu thành công')

Đã đọc dữ liệu thành công


## 2. Khảo sát cấu trúc dataset

Kiểm tra số bản ghi, số thuộc tính, kiểu dữ liệu và thống kê mô tả sơ bộ của dataset.

In [4]:
print('Số dòng, số cột:', df.shape)
df.head()

Số dòng, số cột: (6607, 20)


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Hours_Studied               6607 non-null   int64 
 1   Attendance                  6607 non-null   int64 
 2   Parental_Involvement        6607 non-null   object
 3   Access_to_Resources         6607 non-null   object
 4   Extracurricular_Activities  6607 non-null   object
 5   Sleep_Hours                 6607 non-null   int64 
 6   Previous_Scores             6607 non-null   int64 
 7   Motivation_Level            6607 non-null   object
 8   Internet_Access             6607 non-null   object
 9   Tutoring_Sessions           6607 non-null   int64 
 10  Family_Income               6607 non-null   object
 11  Teacher_Quality             6529 non-null   object
 12  School_Type                 6607 non-null   object
 13  Peer_Influence              6607 non-null   obje

In [6]:
df.describe()

## 3. Danh sách biến (Features)

Dataset gồm 19 biến đầu vào và 1 biến `Exam_Score` (dùng để xây dựng biến mục tiêu `Risk_Level`).

In [7]:
print('Số lượng cột:', len(df.columns))
df.columns.tolist()

Số lượng cột: 20


['Hours_Studied',
 'Attendance',
 'Parental_Involvement',
 'Access_to_Resources',
 'Extracurricular_Activities',
 'Sleep_Hours',
 'Previous_Scores',
 'Motivation_Level',
 'Internet_Access',
 'Tutoring_Sessions',
 'Family_Income',
 'Teacher_Quality',
 'School_Type',
 'Peer_Influence',
 'Physical_Activity',
 'Learning_Disabilities',
 'Parental_Education_Level',
 'Distance_from_Home',
 'Gender',
 'Exam_Score']

**Phân nhóm biến đầu vào:**

| Nhóm | Biến |
|---|---|
| Học tập | Hours_Studied, Attendance, Previous_Scores, Tutoring_Sessions |
| Sinh hoạt | Sleep_Hours, Physical_Activity, Extracurricular_Activities |
| Công nghệ | Internet_Access |
| Tâm lý / Động lực | Motivation_Level |
| Gia đình - Nhà trường - Xã hội | Parental_Involvement, Access_to_Resources, Family_Income, Teacher_Quality, School_Type, Peer_Influence, Learning_Disabilities, Parental_Education_Level, Distance_from_Home, Gender |
| **Mục tiêu (Target)** | Exam_Score (dùng để tạo Risk_Level: Low/Medium/High) |

## 4. Khảo sát dữ liệu thiếu (phát hiện, chưa xử lý)

Kiểm tra số lượng giá trị thiếu ở từng cột. Việc xử lý chi tiết sẽ thực hiện ở Bước 3 - Data Cleaning.

In [8]:
df.isnull().sum()

Hours_Studied                   0
Attendance                      0
Parental_Involvement            0
Access_to_Resources             0
Extracurricular_Activities      0
Sleep_Hours                     0
Previous_Scores                 0
Motivation_Level                0
Internet_Access                 0
Tutoring_Sessions               0
Family_Income                   0
Teacher_Quality                78
School_Type                     0
Peer_Influence                  0
Physical_Activity               0
Learning_Disabilities           0
Parental_Education_Level       90
Distance_from_Home              67
Gender                           0
Exam_Score                       0
dtype: int64

**Phát hiện:** 3 cột có dữ liệu thiếu — Teacher_Quality (78 dòng, ~1.2%), Parental_Education_Level (90 dòng, ~1.4%), Distance_from_Home (67 dòng, ~1.0%). Tỷ lệ đều dưới 2% tổng số dòng.

## 5. Kiểm tra bản ghi trùng lặp (Duplicate)

In [9]:
print('Tổng số dòng:', len(df))
print('Số dòng trùng lặp (duplicate):', df.duplicated().sum())

Tổng số dòng: 6607
Số dòng trùng lặp (duplicate): 0


**Kết quả:** Không phát hiện bản ghi trùng lặp nào (0/6607 dòng).

## 6. Lưu trữ dữ liệu vào MySQL

Sử dụng dịch vụ FreeDB (MySQL online miễn phí) để lưu trữ dữ liệu, đồng thời minh họa khả năng truy vấn SQL trên dữ liệu đã thu thập.

> **Lưu ý:** Password thực tế đã được ẩn để bảo mật thông tin đăng nhập MySQL. Khi chạy lại notebook này, cần thay `YOUR_PASSWORD_HERE` bằng password thật của database FreeDB tương ứng.

In [10]:
!pip install pymysql sqlalchemy --quiet

In [11]:
from sqlalchemy import create_engine

host = 'sql.freedb.tech'
database = 'freedb_7ZmRSeZ0'
username = 'u_nwODG1'
password = 'YOUR_PASSWORD_HERE'

engine = create_engine(f'mysql+pymysql://{username}:{password}@{host}/{database}')
df.to_sql('students', engine, if_exists='replace', index=False)
print('Đã đẩy dữ liệu vào MySQL thành công')

Đã đẩy dữ liệu vào MySQL thành công


## 7. Truy vấn dữ liệu bằng SQL

Thực hiện một số câu truy vấn SQL cơ bản để kiểm tra và khai thác dữ liệu đã lưu trữ.

In [12]:
query_result = pd.read_sql('SELECT COUNT(*) AS total_students FROM students', engine)
print(query_result)

   total_students
0            6607


In [13]:
query_result2 = pd.read_sql('SELECT AVG(Hours_Studied) AS avg_hours, AVG(Attendance) AS avg_attendance FROM students', engine)
print(query_result2)

   avg_hours  avg_attendance
0  19.974266       79.974720


## 8. Kết luận

Dataset gồm 6607 bản ghi, 20 thuộc tính (19 biến đầu vào + 1 biến mục tiêu Exam_Score), không có bản ghi trùng lặp, chỉ 3 cột có dữ liệu thiếu với tỷ lệ dưới 2%. Dữ liệu đã được khảo sát cấu trúc và lưu trữ thành công vào MySQL.

**Bước tiếp theo:** `02_data_cleaning.ipynb` - Làm sạch dữ liệu.